# Script 18 — Unique variance per predictor, with a permutation reference (Supp Fig S6)

For each supportive regression of a biological measure (KDM-BA advancement,
log SII) on one rhythm metric plus the core covariates, within each age group
(18-59, 60+), this notebook computes:

1. the model R²;
2. the unique variance of each predictor, the squared semi-partial
   correlation: the drop in R² when that predictor alone is removed. Unique
   contributions do not sum to the model R²; the gap is variance shared among
   correlated predictors;
3. a permutation reference for the rhythm metric: the observed metric is
   permuted across participants within the equation sample 5,000 times (seed
   20260828), with covariates, outcome and weights fixed, and the increment
   recomputed each time. This preserves the metric's marginal distribution
   and removes its correspondence with the biological measure. A
   Freedman-Lane variant (permuting the residual of X | C) agreed with this
   scheme to within Monte Carlo noise in all twelve cells.

The reference is descriptive: it calibrates the size of an increment and
carries no p value. Survey-weighted least squares with the four-year weight in
the point estimate only, matching scripts 7, 8 and 17; these are
decompositions of the supportive regressions, not of the structural equation
models.

Reads `data/processed/8_nhanes_analytic.csv`.
Writes `figure/supplementary/SuppFigS6_Variance_Decomposition.{pdf,png}`
and `figure/source_data/Source_Data_SuppFigS6_Variance_Decomposition.xlsx`.
Runtime about 1 minute (5,000 permutations × 12 cells).

## Step 0 — Imports, paths, constants

In [ ]:
import os, sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
warnings.filterwarnings('ignore')
pd.options.display.max_columns = 60

def find_root():
    """Locate the project root without assuming where the kernel started."""
    env = os.environ.get('NHANES_SEM_ROOT')
    cands = ([Path(env)] if env else []) + [Path.cwd(), *Path.cwd().parents]
    for q in cands:
        if (q / 'data' / 'processed').is_dir() and (q / 'scripts').is_dir():
            return q
    raise RuntimeError(
        'cannot find the project root from ' + str(Path.cwd())
        + ' -- start Jupyter inside the project, or set NHANES_SEM_ROOT')

BASE     = find_root()
ANALYTIC = BASE / 'data' / 'processed' / '8_nhanes_analytic.csv'
FIG_SUPP = BASE / 'figure' / 'supplementary'
SRC      = BASE / 'figure' / 'source_data'
for q in (FIG_SUPP, SRC):
    q.mkdir(parents=True, exist_ok=True)
STEM = 'SuppFigS6_Variance_Decomposition'

sys.path.insert(0, str(BASE / 'scripts'))

# Reload project modules so the notebook runs against the code on disk.
import importlib
import wls_utils, nature_palette
import population; importlib.reload(population)
for _m in (wls_utils, nature_palette,):
    importlib.reload(_m)

from wls_utils import wls_fit
from population import ADULT_MIN_AGE, OLD_MIN_AGE, YOUNG, OLD, POOLED
from nature_palette import EXP_COL, EXP_ORDER, quiet_fonttools

# Same figure style as mplus_report, copied so that this notebook does not
# import the Mplus reporting chain; verify_figures.py checks fonts, dpi, width
# and source data.
quiet_fonttools()
matplotlib.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Helvetica', 'Arial', 'DejaVu Sans'],
    'font.size': 8, 'axes.labelsize': 8, 'axes.titlesize': 8,
    'xtick.labelsize': 7, 'ytick.labelsize': 7, 'legend.fontsize': 7,
    'axes.linewidth': 0.5, 'xtick.major.width': 0.5, 'ytick.major.width': 0.5,
    'xtick.major.size': 2.5, 'ytick.major.size': 2.5,
    'lines.linewidth': 0.9, 'patch.linewidth': 0.5,
    'pdf.fonttype': 42, 'ps.fonttype': 42,
    'mathtext.fontset': 'dejavusans', 'mathtext.default': 'regular',
    'axes.spines.top': False, 'axes.spines.right': False,
})

RENAME = {'RIDAGEYR': 'AGE', 'RIAGENDR': 'SEX', 'BMXBMI': 'BMI',
          'DMDEDUC2': 'EDU', 'Wear_Time': 'WEAR_T', 'SLD010H': 'SLEEP_H',
          'KDM_BA_advance': 'BA_ADV', 'SII_log': 'SII_LOG', 'dpq_sum': 'PHQ9'}
COVS = ['AGE', 'SEX', 'BMI', 'EDU', 'WEAR_T', 'SLEEP_H']
COV_LABEL = {'AGE': 'Age', 'SEX': 'Sex', 'BMI': 'Body mass index',
             'EDU': 'Education', 'WEAR_T': 'Wear time', 'SLEEP_H': 'Sleep duration'}
# Muted covariate greys, ordered so that body mass index (the dominant
# covariate) is adjacent to the metric block.
COV_COL = {'BMI': '#6E7B8B', 'EDU': '#98A6B3', 'SEX': '#BFC8D2',
           'SLEEP_H': '#D5DBE1', 'AGE': '#E4E8EC', 'WEAR_T': '#F0F2F4'}
COV_DRAW = ['BMI', 'EDU', 'SEX', 'SLEEP_H', 'AGE', 'WEAR_T']

DEPS  = [('BA_ADV', 'KDM-BA advancement'), ('SII_LOG', 'log SII')]
CELLS = [(YOUNG, ADULT_MIN_AGE, OLD_MIN_AGE), (OLD, OLD_MIN_AGE, 999)]
N_DRAWS = 5000        # permutations per predictor per equation
SEED    = 20260828

MM = 1 / 25.4
W2 = 183 * MM

print('root      ', BASE)
print('analytic  ', ANALYTIC.name, '-- exists:', ANALYTIC.exists())
print(f'{N_DRAWS:,} permutations x {len(DEPS) * len(CELLS) * len(EXP_ORDER)} cells')

## Step 1 — The decomposition

`decompose()` returns, for one (dependent variable × metric × age group) cell:
`model_r2`; one entry per predictor (`full − reduced` R² for that predictor
alone); and `null_median` / `null_p95` / `null_percentile_of_actual` from the
5,000 permutations. The reduced (covariates-only) model is computed once; only
the full model is refitted per draw.

In [ ]:
def _r2(X, y, w):
    return wls_fit(X, y, w=w, se='hc3', add_const=True)['r2']


def decompose(d, dep, exposure, lo, hi, rng):
    """Unique variance per predictor, model R-square, and the permutation reference."""
    need = [exposure, dep, 'WTMEC4YR'] + COVS
    s = d[(d.AGE >= lo) & (d.AGE < hi)]
    s = s[s[need].notna().all(axis=1)]
    if len(s) < 200:
        return None
    w = s.WTMEC4YR.to_numpy(float)
    y = s[dep].to_numpy(float)
    names = [exposure] + COVS
    M = np.column_stack([s[k].to_numpy(float) for k in names])
    full = _r2(M, y, w)

    out = {'model_r2': full, 'n': len(s)}
    for i, k in enumerate(names):
        out[k] = full - _r2(np.delete(M, i, axis=1), y, w)

    # Permutation reference: the observed metric is permuted across
    # participants; covariates, outcome and weights stay fixed. The metric's
    # marginal distribution is preserved and its pairing with the outcome is
    # broken.
    C_ = M[:, 1:]
    x_obs = M[:, 0]
    reduced = _r2(C_, y, w)
    null = np.array([_r2(np.column_stack([rng.permutation(x_obs), C_]), y, w)
                     - reduced for _ in range(N_DRAWS)])
    out['null_median'] = float(np.median(null))
    out['null_p95'] = float(np.percentile(null, 95))
    out['null_percentile_of_actual'] = float((null < out[exposure]).mean() * 100)
    out['n_draws'] = N_DRAWS
    return out


def build():
    d = pd.read_csv(ANALYTIC, low_memory=False).rename(columns=RENAME)
    d = d[d.AGE >= ADULT_MIN_AGE]
    rng = np.random.default_rng(SEED)
    rows = []
    for dep, dep_lab in DEPS:
        for cell_, lo, hi in CELLS:
            for exp in EXP_ORDER:
                r = decompose(d, dep, exp, lo, hi, rng)
                if r is None:
                    continue
                rows.append(dict(dependent=dep_lab, age_group=cell_, metric=exp,
                                 n=r['n'], model_r2=r['model_r2'],
                                 unique_rhythm=r[exp],
                                 **{f'unique_{c.lower()}': r[c] for c in COVS},
                                 null_median=r['null_median'],
                                 null_p95=r['null_p95'],
                                 null_percentile_of_actual=r['null_percentile_of_actual'],
                                 n_draws=r['n_draws']))
    return pd.DataFrame(rows)

## Step 2 — Run

About one minute; the permutation loop is the cost.

In [ ]:
t = build()
assert len(t) == len(DEPS) * len(CELLS) * len(EXP_ORDER), \
    f'expected {len(DEPS) * len(CELLS) * len(EXP_ORDER)} cells, got {len(t)}'
t[['dependent', 'age_group', 'metric', 'n', 'model_r2', 'unique_rhythm',
   'null_median', 'null_p95', 'null_percentile_of_actual']]

## Step 3 — The figure

Stacked bars of unique variance per predictor, one bar per rhythm metric,
grouped by age group. A solid line marks the model R² and a dashed line the
95th percentile of that metric's own permutation reference.

In [ ]:
def draw(t):
    fig, axes = plt.subplots(1, 2, figsize=(W2, 90 * MM), sharey=True)
    for ax, (dep, dep_lab) in zip(axes, DEPS):
        sub = t[t.dependent == dep_lab]
        labels, ticks, spans = [], [], []
        x = 0
        for cell_, _, _ in CELLS:
            span_start = x
            blk = sub[sub.age_group == cell_]
            for exp in EXP_ORDER:
                r = blk[blk.metric == exp]
                if not len(r):
                    continue
                r = r.iloc[0]
                bottom = 0
                ax.bar(x, r.unique_rhythm, 0.72, bottom=bottom,
                       color=EXP_COL[exp], lw=0)
                bottom += r.unique_rhythm
                for c in COV_DRAW:
                    v = r[f'unique_{c.lower()}']
                    ax.bar(x, v, 0.72, bottom=bottom, color=COV_COL[c], lw=0)
                    bottom += v
                ax.plot([x - 0.36, x + 0.36], [r.model_r2] * 2,
                        color='0.15', lw=0.9, solid_capstyle='butt')
                ax.plot([x - 0.36, x + 0.36], [r.null_p95] * 2,
                        color='#B03030', lw=0.9, ls=(0, (2, 1.5)),
                        solid_capstyle='butt', zorder=5)
                ticks.append(x); labels.append(exp)
                x += 1
            spans.append((cell_, (span_start + x - 1) / 2))
            x += 1.4
        ax.set_xticks(ticks)
        ax.set_xticklabels(labels, fontsize=7)
        ax.set_title({'KDM-BA advancement': 'KDM-BA advancement',
                      'log SII': 'Log SII'}.get(dep_lab, dep_lab), pad=3, loc='left')
        ax.set_ylim(0, max(t.model_r2) * 1.12)
        for cell_, xmid in spans:
            ax.annotate(('\u2265' + cell_.rstrip('+') if cell_.endswith('+') else cell_.replace('-', '\u2013')) + ' years', xy=(xmid, 0),
                        xycoords=('data', 'axes fraction'),
                        xytext=(0, -16), textcoords='offset points',
                        ha='center', va='top', fontsize=7, color='0.35')
    axes[0].set_ylabel('Variance explained (%)')
    axes[0].yaxis.set_major_formatter(lambda v, _: f'{v*100:.0f}')

    h = [Patch(facecolor=EXP_COL[e], label={'RA': 'Relative amplitude (RA)',
                                            'IS': 'Interdaily stability (IS)',
                                            'IV': 'Intradaily variability (IV)'}[e])
         for e in EXP_ORDER]
    h += [Patch(facecolor=COV_COL[c], label=COV_LABEL[c]) for c in COV_DRAW]
    h += [plt.Line2D([], [], color='0.15', lw=0.9,
                     label='Full-model R\u00b2'),
          plt.Line2D([], [], color='#B03030', lw=0.9, ls=(0, (2, 1.5)),
                     label='95th percentile of rhythm-permutation reference')]
    fig.legend(handles=h, loc='lower center', ncol=3, frameon=False,
               bbox_to_anchor=(0.5, -0.17), fontsize=7)
    fig.tight_layout()
    for ax, letter in zip(axes, 'ab'):   # lower-case panel letters (EJE)
        fig.text(ax.get_position().x0 - 0.045, 0.985, letter, ha='left',
                 va='top', fontsize=10, fontweight='bold')
    return fig


fig = draw(t)
fig

## Step 4 — Write the figure and its source data

In [ ]:
CAPTION = [
    'Supplementary Fig. S6 | Incremental variance explained by each predictor '
    'in the equations for the two biological measures.',
    'a, KDM biological-age advancement; b, log systemic immune-inflammation '
    'index (SII); one bar per rhythm metric within each age group (20-59 and '
    '\u226560 years).',
    'Each bar segment is the incremental R\u00b2 of one predictor: the reduction '
    'in R\u00b2 when that predictor alone is removed from the survey-weighted '
    'least-squares equation (squared semi-partial correlation). The coloured '
    'bottom segment is the rhythm metric; the grey segments are the covariates. '
    'The black line is the R\u00b2 of the full model. Increments obtained by '
    'removing one predictor at a time do not in general add up to the '
    'full-model R\u00b2; no formal commonality decomposition was done, so the '
    'distance between the top of a bar and the black line is not attributed to '
    'any predictor or set of predictors.',
    'The dashed red line applies to the rhythm segment only: the 95th '
    'percentile of the rhythm increment when the metric\'s values were randomly '
    'permuted across participants 5,000 times, with the biological measure, the '
    'covariates and the survey weights held fixed. It is a scale reference for '
    'the rhythm segment, not a significance test, and is not a reference for '
    'the covariate segments; the percentile of each observed increment within '
    'its own reference is in the source data.',
    'Chronological age contributes little to panel a because the dependent '
    'variable is biological minus chronological age.',
    'Survey-weighted least squares with the four-year weight in the point '
    'estimate; descriptive decompositions of the supportive regressions, not '
    'of the structural equation models.',
]

fig.savefig(FIG_SUPP / f'{STEM}.pdf', bbox_inches='tight')
fig.savefig(FIG_SUPP / f'{STEM}.png', bbox_inches='tight', dpi=600)

with pd.ExcelWriter(SRC / f'Source_Data_{STEM}.xlsx', engine='openpyxl') as xw:
    t.to_excel(xw, sheet_name='source data', index=False)
    pd.DataFrame({'Notes': CAPTION}).to_excel(xw, sheet_name='legend', index=False)

print(f'wrote {(FIG_SUPP / STEM).relative_to(BASE)}.pdf + .png and its source data')

## Step 5 — Read the result

`<- within permutation range` marks a cell whose observed increment does not exceed the 95th percentile of its own reference.

In [ ]:
for lab in ['KDM-BA advancement', 'log SII']:
    print(f'\n{lab}:')
    for _, r in t[t.dependent == lab].iterrows():
        flag = '  <- within permutation range' if r.unique_rhythm <= r.null_p95 else ''
        print(f'  {r.metric:3s} {r.age_group:6s} unique {r.unique_rhythm*100:6.3f}%'
              f'   null median {r.null_median*100:.3f}%  p95 {r.null_p95*100:.3f}%'
              f'   percentile {r.null_percentile_of_actual:5.1f}{flag}')